# 四种化合物浓度核验（独立审计）
范围：DMS、双乙酰、4VG、反式-2-壬烯醛；重点 beer=73、156、34、151。
仅读取原始文件，不删异常值、不填补、不替换来源、不训练或选择模型、不生成风格。
附件1是混合尺度，不能整表视为浓度。相对0–6级是训练集分位数分级，真实原始评分还原尚未解决。
本笔记本保存实际运行输出；所有数值判断均来自本地附件，不补造外部阈值。

In [1]:
from pathlib import Path
import hashlib, json, sys
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / 'data').is_dir() else Path.cwd().parent
DATA = ROOT / 'data'
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 100)
pd.set_option('display.precision', 12)
existing = sorted(p for folder in ['data','notebooks','outputs'] for p in (ROOT/folder).rglob('*') if p.is_file() and p.name != '05_defect_concentration_audit.ipynb')
before = {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in existing}
display(pd.DataFrame([{'path': k, 'bytes': (ROOT/k).stat().st_size, 'sha256': v} for k,v in before.items()]))
print('Python:',sys.version)
print('数据说明：\n'+(DATA/'数据说明.txt').read_text(encoding='utf-8'))
print('赛题文件：', [str(p.relative_to(ROOT)) for p in ROOT.rglob('*') if p.is_file() and '.venv' not in p.parts and (p.suffix.lower() in ['.pdf','.docx'] or '赛题' in p.name)])


,path,bytes,sha256
0,data\1.csv,700705,419849227a02e0c792cd405e6865035b7f58af1a92efa3...
1,data\2.csv,1000771,415709ac10f2f730d31f249bd6ac108250bdfbfd62e005...
2,data\3.xlsx,163364,fe4dcabd48431e6d677c095962ea9c447e10e509c4f3f8...
3,data\4.csv,162887,43a7b38682ed81ce64d97ebc1874c09dffac023ade4877...
4,data\5.csv,48044,c512eed108de6f98e0c6ed66497ed2319745d30ca7f6f4...
5,data\6.csv,42937,a55004fb0390299a5c8089f0c37d3bfc006e7049963c84...
6,data\7.csv,588816,2083138df46640e20b46f7e76ec32cf9b380fd34e01926...
7,data\8.xlsx,1740092,f2316766792111d98281bd35c8403682be8c8069d166a7...
8,data\数据说明.txt,614,40813e273e081ae0dc6d6fa76838c5cdbaa721335df364...
9,notebooks\01_data_check.ipynb,174978,42c9e036ea7bca8e7777db366111edb905e3bd58362e54...


Python: 3.13.0 (tags/v3.13.0:60403a5, Oct  7 2024, 09:38:07) [MSC v.1941 64 bit (AMD64)]
数据说明：
文件名1
说明：全部 250 个啤酒样品的化学测量数据。
文件名2
说明：所有化学参数之间的 Spearman 相关系数得分。
文件名3
说明：感官分析过程中由经过培训的品评小组使用的品评表。
文件名4
说明：全部 250 个啤酒样品的感官分析结果。
文件名5
说明：所有感官参数之间的 Spearman 相关系数得分。
文件名6
说明：化学参数与感官参数之间的 Spearman 相关系数。
文件名7
说明：针对全部 250 个啤酒样品的感官描述符的论坛评论数据。
文件名8
其他相关数据


赛题文件： []


## 已有材料与证据边界
当前工作区没有找到独立赛题正文（PDF、DOCX或文件名含“赛题”的文件），只能核对数据说明、README、已有 Notebook 和附件，不能声称已核验赛题原文。
01_data_check 第48–52单元已做初步变换、差异及阈值比检查；其中最后一格使用19、0.09、0.20作为阈值。下文核对它们的来源，不沿用其风险分类。
02_prediction、04_style_generation 当前为空白占位；03_compound_analysis 为已有关联分析。保留全部原文件。

In [2]:
for p in sorted((ROOT/'notebooks').glob('*.ipynb')):
    if p.name == '05_defect_concentration_audit.ipynb': continue
    old = json.loads(p.read_text(encoding='utf-8'))
    print(p.name, 'cells=',len(old['cells']), 'saved_outputs=',sum(len(c.get('outputs',[])) for c in old['cells']))
    hits = [(i+1, ''.join(c['source'])) for i,c in enumerate(old['cells']) if any(t in ''.join(c['source']).lower() for t in ['threshold','nonenal','butanedione','check_features'])]
    for i,s in hits: print('相关单元',i, s[:6000])
print('README:'); print((ROOT/'README.md').read_text(encoding='utf-8'))
display(pd.DataFrame([{'file': p.name, 'rows':len(pd.read_csv(p)), 'columns':len(pd.read_csv(p).columns)} for p in sorted((ROOT/'outputs').glob('*.csv'))]))


01_data_check.ipynb cells= 52 saved_outputs= 54
相关单元 48 calibrated = pd.read_excel(
    data_dir / "8.xlsx",
    sheet_name="Calibrated chemical dataset"
)

assert chem["beer"].is_unique
assert calibrated["beer"].is_unique

chem_by_beer = chem.set_index("beer")
cal_by_beer = calibrated.set_index("beer")

check_features = ["DMS", "X2.3.butanedione", "X4VG"]
check_rows = []

for feature in check_features:
    paired = pd.concat([
        chem_by_beer[feature].rename("附件1"),
        cal_by_beer[feature].rename("校准浓度")
    ], axis=1, join="inner").dropna()

    # 对数只对正浓度计算
    positive = paired.loc[paired["校准浓度"] > 0]
    observed = positive["附件1"]
    concentration = positive["校准浓度"]

    candidates = {
        "原值": concentration,
        "自然对数ln": np.log(concentration),
        "常用对数log10": np.log10(concentration)
    }

    for name, candidate in candidates.items():
        difference = (observed - candidate).abs()

        check_rows.append({
            "化学字段": feature,
            "

,file,rows,columns
0,all_candidates_top3_frequency.csv,218,31
1,candidate_style_sensitivity.csv,150,6
2,chemical_name_mapping.csv,231,12
3,compound_sensory_correlations.csv,6540,5
4,feature_roles.csv,231,2
5,model_selection_cv.csv,30,5
6,model_selection_cv_50.csv,50,5
7,network_edges_bootstrap.csv,245,5
8,network_edges_from_official.csv,245,4
9,network_edges_stable.csv,172,5


In [3]:
a = pd.read_csv(DATA/'1.csv')
b = pd.read_excel(DATA/'8.xlsx',sheet_name='Calibrated chemical dataset')
t = pd.read_excel(DATA/'8.xlsx',sheet_name='Sensory thresholds')
fields = ['DMS','X2.3.butanedione','X4VG','X.E..2.nonenal']
assert a.beer.notna().all() and b.beer.notna().all()
assert a.beer.is_unique and b.beer.is_unique
assert set(a.beer) == set(b.beer) and len(a)==len(b)==250
a=a.set_index('beer'); b=b.set_index('beer').reindex(a.index)
assert a.beer_id.equals(b.beer_id)
print('附件1/8形状',a.shape,b.shape,'；250个beer唯一且集合相同，beer_id逐项一致')
t['field_name_R'] = t['field_name_R'].astype('string').str.strip()
meta=t.loc[t.field_name_R.isin(fields)]
assert len(meta)==4 and meta.field_name_R.is_unique
display(meta)
print('附件3工作表：', pd.ExcelFile(DATA/'3.xlsx').sheet_names)
print('附件8工作表：', pd.ExcelFile(DATA/'8.xlsx').sheet_names)


附件1/8形状 (250, 233) (250, 72) ；250个beer唯一且集合相同，beer_id逐项一致


,field_name_python,field_name_R,chemical_name,chemical_name_short,calibrated,origin_paper,unit,aging,off_flavors,threshold_mean,threshold_median
53,(E)-2-nonenal,X.E..2.nonenal,E-2-nonenal,E-2-nonenal,yes_gcms,Others,mg/L,increase,NaN,NaN,NaN
125,"2,3-butanedione",X2.3.butanedione,Diacetyl,Diacetyl,yes_gcms,Yeast,mg/L,increase,"butterscotch, popcorn, oily, milky",0.299250000,0.09
144,X4VG,X4VG,4-vinylguaiacol,4-vinylguaiacol,yes_gcms,Yeast,mg/L,NaN,NaN,0.206333333,0.20
153,DMS,DMS,Dimethyl sulfide,Dimethyl sulfide,yes,Malt,ug/L,increase,"cooked vegetables, onion, garlic, corn, black ...",17.250000000,19.00


附件3工作表： ['beer_1', 'beer_2', 'beer_3', 'beer_4', 'beer_5', 'beer_6', 'beer_7', 'beer_8', 'beer_9', 'beer_10', 'beer_11', 'beer_12', 'summary']
附件8工作表： ['Sensory thresholds', 'Sensory names', 'GBM importance MDI', 'GBM SHAP Values', 'Calibrated chemical dataset', 'Beer Prices', 'Sheet2']


## 单位、零值、缺失与候选变换
单位来自附件8 Sensory thresholds 的 unit 字段：DMS为ug/L，其余三种为mg/L。这是附件元数据声明，不是独立实验校准证明；附件1的log值本身不能直接与浓度阈值比较。
只在附件8严格正值且两侧非缺失的配对上比较原值、ln、log10、log1p，另行完整列出零值及缺失。绝对误差容限1e-6，rtol=0；“匹配”仅表示数值关系。

In [4]:
stats=[]; transforms=[]; issues=[]
for f in fields:
    x,y=a[f],b[f]
    for src,s in [('附件1',x),('附件8',y)]:
        stats.append(dict(field=f,source=src,n=len(s),missing=int(s.isna().sum()),zero=int(s.eq(0).sum()),negative=int(s.lt(0).sum()),infinite=int(np.isinf(s).sum()),minimum=s.min(),maximum=s.max()))
    mask=x.notna() & y.notna() & y.gt(0)
    for name,z in [('identity',y),('ln',np.log(y.where(y>0))),('log10',np.log10(y.where(y>0))),('log1p',np.log1p(y.where(y>=0)))]:
        delta=(x[mask]-z[mask]).abs()
        transforms.append(dict(field=f,transform=name,n=int(mask.sum()),matched=int(delta.le(1e-6).sum()),MAE=delta.mean(),max_abs_error=delta.max()))
    selected=x.isna()|y.isna()|x.eq(0)|y.eq(0)
    for beer in a.index[selected]: issues.append(dict(field=f,beer=beer,attachment1=x.loc[beer],attachment8=y.loc[beer],a1_missing=pd.isna(x.loc[beer]),a8_missing=pd.isna(y.loc[beer])))
display(pd.DataFrame(stats)); display(pd.DataFrame(transforms))
issues=pd.DataFrame(issues)
display(issues)


,field,source,n,missing,zero,negative,infinite,minimum,maximum
0,DMS,附件1,250,0,0,0,0,0.046211200,2.716380377
1,DMS,附件8,250,0,1,0,0,0.000000000,520.451635000
2,X2.3.butanedione,附件1,250,9,0,0,0,0.000000059,121.731018200
3,X2.3.butanedione,附件8,250,0,9,0,0,0.000000000,121.731018246
4,X4VG,附件1,250,0,0,250,0,-2.013157381,-0.164417223
5,X4VG,附件8,250,0,168,0,0,0.000000000,0.684830000
6,X.E..2.nonenal,附件1,250,0,0,0,0,0.000061400,14.910091700
7,X.E..2.nonenal,附件8,250,1,0,0,0,0.000061350,14.910091698


,field,transform,n,matched,MAE,max_abs_error
0,DMS,identity,249,0,32.842764256877,517.735254623000
1,DMS,ln,249,0,1.821529180123,3.538316586469
2,DMS,log10,249,249,0.000000000281,0.000000000498
3,DMS,log1p,249,0,1.872234602939,3.540236151050
4,X2.3.butanedione,identity,241,238,0.002690617089,0.311015820000
5,X2.3.butanedione,ln,241,0,4.765990690914,116.929204357823
6,X2.3.butanedione,log10,241,0,3.623171177323,119.645616945216
7,X2.3.butanedione,log1p,241,23,2.278431285128,116.921023082790
8,X4VG,identity,82,0,1.109546013078,1.731530551700
9,X4VG,ln,82,0,1.260622901792,2.230191609275


,field,beer,attachment1,attachment8,a1_missing,a8_missing
0,DMS,10,0.046211200,0.0,False,False
1,X2.3.butanedione,33,NaN,0.0,True,False
2,X2.3.butanedione,44,NaN,0.0,True,False
3,X2.3.butanedione,223,NaN,0.0,True,False
4,X2.3.butanedione,26,NaN,0.0,True,False
...,...,...,...,...,...,...
174,X4VG,175,-2.013157381,0.0,False,False
175,X4VG,196,-2.013157381,0.0,False,False
176,X4VG,4,-2.013157381,0.0,False,False
177,X4VG,99,-2.013157381,0.0,False,False


In [5]:
zero_evidence=[]
for f in ['DMS','X4VG']:
    x,y=a[f],b[f]
    zero=y.eq(0); positive=y.gt(0)
    floor=x[zero].unique()
    recovered=np.power(10.0,floor)
    minpos=y[positive].min()
    zero_evidence.append(dict(field=f,n_zero=int(zero.sum()),a1_zero_codes=floor.tolist(),inverse_log10=recovered.tolist(),a8_min_positive=minpos,half_min_positive=minpos/2,positive_shares_zero_code=int(x[positive].isin(floor).sum())))
display(pd.DataFrame(zero_evidence))


,field,n_zero,a1_zero_codes,inverse_log10,a8_min_positive,half_min_positive,positive_shares_zero_code
0,DMS,1,[0.0462112],[1.112272500539],2.2245450000,1.11227250000,0
1,X4VG,168,[-2.013157381],[0.009701583345],0.0194031667,0.00970158335,0


DMS的249个正值、4VG的82个正值均符合log10（最大误差约5e-10）。附件8零值分别1个、168个；附件1对应固定有限值，逆log10约等于该列最小正浓度的一半。这支持“零值经半最小正值替代后取log10”的数据关系，但不能证明生成脚本或零值的实验含义；不能据此认定为真实绝对零、低于检出限或缺失。原数据保持不变。

## 原值对应字段的全部差异（含缺失状态差异）

In [6]:
differences=[]
for f in ['X2.3.butanedione','X.E..2.nonenal']:
    same=np.isclose(a[f],b[f],atol=1e-6,rtol=0,equal_nan=True)
    for beer in a.index[~same]:
        x,y=a.at[beer,f],b.at[beer,f]
        differences.append(dict(field=f,beer=beer,attachment1=x,attachment8=y,delta_a1_minus_a8=x-y,kind='missing_state' if pd.isna(x) or pd.isna(y) else 'numeric_difference'))
diff=pd.DataFrame(differences)
display(diff)
focus = pd.concat([a.loc[[73,156,34,151],fields].assign(source='附件1'),b.loc[[73,156,34,151],fields].assign(source='附件8')]).reset_index()
display(focus)
display(b[['beer_id','tasting_category_fine','X2.3.butanedione']].nlargest(10,'X2.3.butanedione'))
print('beer=73 / 次高双乙酰：',b.at[73,'X2.3.butanedione']/b['X2.3.butanedione'].nlargest(2).iloc[1])


,field,beer,attachment1,attachment8,delta_a1_minus_a8,kind
0,X2.3.butanedione,33,NaN,0.000000000,NaN,missing_state
1,X2.3.butanedione,44,NaN,0.000000000,NaN,missing_state
2,X2.3.butanedione,223,NaN,0.000000000,NaN,missing_state
3,X2.3.butanedione,26,NaN,0.000000000,NaN,missing_state
4,X2.3.butanedione,151,0.139752287,0.043173294,0.096578993,numeric_difference
5,X2.3.butanedione,156,0.434129935,0.123114115,0.311015820,numeric_difference
6,X2.3.butanedione,176,NaN,0.000000000,NaN,missing_state
7,X2.3.butanedione,129,NaN,0.000000000,NaN,missing_state
8,X2.3.butanedione,119,NaN,0.000000000,NaN,missing_state
9,X2.3.butanedione,28,NaN,0.000000000,NaN,missing_state


,beer,DMS,X2.3.butanedione,X4VG,X.E..2.nonenal,source
0,73,1.498849503,121.731018200,-0.838841708,0.0090705160,附件1
1,156,1.313158806,0.434129935,-2.013157381,0.0246385010,附件1
2,34,1.312594308,0.289621935,-2.013157381,10.9595130500,附件1
3,151,1.175334348,0.139752287,-2.013157381,0.0101369440,附件1
4,73,31.539115000,121.731018246,0.144930000,0.0090705155,附件8
5,156,20.566425000,0.123114115,0.000000000,0.0246385010,附件8
6,34,20.539710000,0.530465527,0.000000000,10.9595130530,附件8
7,151,14.973880000,0.043173294,0.000000000,0.0101369435,附件8


,beer_id,tasting_category_fine,X2.3.butanedione
beer,,,
73,5410769100081,Blond,121.7310182460
19,5425030060032,Amber,92.8378317780
118,5411364151119,West Flanders ale,88.5289896095
9,5411081005368,Hoppy,86.4253136320
158,54107014,Amber,79.3617841240
177,5425100322608,Fruitbeer,54.9517134600
178,620670293948,West Flanders ale,5.2945283920
66,5413977000037,Tripel,5.2527764550
80,5415152000313,Hoppy,3.7641844310


beer=73 / 次高双乙酰： 1.3112221161852564


## 回到Excel原始单元格核对
核对源单元格地址、值、类型、公式/批注/格式，排除按行号读取和明显显示格式误解。没有公式或批注不代表数值已经通过实验质量控制。

In [7]:
wb=load_workbook(DATA/'8.xlsx',data_only=False,read_only=False)
ws=wb['Calibrated chemical dataset']
headers={c.value:c.column for c in ws[1]}
rows={ws.cell(r,headers['beer']).value:r for r in range(2,ws.max_row+1)}
records=[]
for beer in [73,156,34,151,158,130,177]:
    for f in fields:
        c=ws.cell(rows[beer],headers[f])
        records.append(dict(beer=beer,field=f,cell=c.coordinate,value=c.value,data_type=c.data_type,number_format=c.number_format,comment=c.comment.text if c.comment else None))
display(pd.DataFrame(records))
wb.close()


,beer,field,cell,value,data_type,number_format,comment
0,73,DMS,G219,31.539115,n,General,None
1,73,X2.3.butanedione,AO219,121.731018246,n,General,None
2,73,X4VG,X219,0.14493,n,General,None
3,73,X.E..2.nonenal,BI219,0.0090705155,n,General,None
4,156,DMS,G75,20.566425,n,General,None
5,156,X2.3.butanedione,AO75,0.123114115,n,General,None
6,156,X4VG,X75,0,n,General,None
7,156,X.E..2.nonenal,BI75,0.024638501,n,General,None
8,34,DMS,G192,20.53971,n,General,None
9,34,X2.3.butanedione,AO192,0.530465527,n,General,None


## 可确定的结论与待决定事项

- **单位（附件声明）**：DMS ug/L；双乙酰、4VG、反式-2-壬烯醛 mg/L。DMS换算mg/L需除1000，但本次不改变任何值。
- **双乙酰**：241个共同非缺失值中238个按原值匹配，3个不匹配：beer156约为0.434130 / 0.123114；beer34约为0.289622 / 0.530466；beer151约为0.139752 / 0.043173（附件1 / 附件8）（精确值以上方运行输出为准）。附件1有9个缺失，附件8这9个位置全为0，不能自动当作已测得0，也不能直接判断哪份正确。
- **beer73极端值**：两份附件均为约121.731018 mg/L，附件8源格为数字而非公式。它是全列最高值，但另有数个高浓度样本；附件间一致不构成测量真实性证明。没有证据支持擅自除1000、删行或缩尾。
- **反式-2-壬烯醛**：249个共同有效值中247个按原值匹配。beer158、130数值不同；beer177附件1有值、附件8缺失。两份均无0，附件1无缺失，附件8有1个缺失。不能静默合并。
- **阈值溯源**：已有代码的19、0.09、0.20恰为附件8三项threshold_median；对应mean为17.25、0.29925、0.206333…。本次只核对来源，不把感知阈值直接定为缺陷判定阈值。反式-2-壬烯醛的mean、median均缺失，不补造。
- **下一步需决定**：差异样本采用哪一来源或维持冲突标记；9个“缺失/0”及DMS、4VG零值的实验语义；beer73及其他高值是否需原研究勘误或检测记录确认；缺陷定义、风格适用性及阈值依据。未解决前不做第三问风格生成模型。

以上小数以代码显示的源值为准；推断与已证实的数值关系分开。赛题正文未在当前工作区找到，需后续补齐才能核查题目对单位或阈值的额外限定。

In [8]:
after={k:hashlib.sha256((ROOT/k).read_bytes()).hexdigest() for k in before}
assert before == after, '既有文件发生改变'
assert ((diff.field=='X2.3.butanedione') & (diff.kind=='numeric_difference')).sum()==3
assert set(diff.loc[(diff.field=='X2.3.butanedione') & (diff.kind=='numeric_difference'),'beer']) == {156,34,151}
print(f'核验完成：{len(before)}个既有数据、Notebook及输出文件的SHA256均未改变。未填补、未删行、未训练模型。')


核验完成：33个既有数据、Notebook及输出文件的SHA256均未改变。未填补、未删行、未训练模型。
